# Beyond BLAST Notebook

In [2]:
ENV["JULIA_PKG_PRECOMPILE_AUTO"] = 0;
using Pkg
Pkg.activate("blast_code"; io=devnull)
Pkg.resolve(; io=devnull)
Pkg.instantiate(; io=devnull)

using Revise 

using Base.Threads, NPZ, DataInterpolations, Interpolations, FastChebInterp
using BenchmarkTools, FFTW, FastTransforms, Dates, TOML, Plots, Plots.Measures
using QuadGK, LaTeXStrings, Tullio, StaticArrays, LoopVectorization, LinearAlgebra
using Unitful, SpecialFunctions, DifferentialEquations, Cosmology, NumericalIntegration
using CSV, DataFrames, JSON, OrderedCollections
;

In [11]:
include("blast_code/src/Blast.jl")
using .Blast

#### Including the .jl modules

In [21]:
using SpecialFunctions

# 1. Funzione di Bessel e Trapezio (corretta senza @. nel loop)
j_l(l, x) = sphericalbesselj(l, x)

function trapz(x, y)
    n = length(x)
    integral = 0.0
    @inbounds for i in 1:(n-1)
        # Rimosso il @. qui, dato che x[i] e y[i] devono essere scalari
        integral += 0.5 * (x[i+1] - x[i]) * (y[i+1] + y[i])
    end
    return integral
end

function calc_I_l(k, k_i, l, chi_grid, W_chi_array)
    integrand = @. W_chi_array * j_l(l, k * chi_grid) * j_l(l, k_i * chi_grid)
    return trapz(chi_grid, integrand)
end

# 2. Funzione principale modificata per accettare l'array P(k) direttamente
function brute_force_S_lkk_gg(k1, k2, l, P_k_array, chi_grid, W_chi_array, k_grid)
    I_k_k1 = zeros(length(k_grid))
    I_k_k2 = zeros(length(k_grid))
    
    Threads.@threads for i in eachindex(k_grid)
        k_val = k_grid[i]
        I_k_k1[i] = calc_I_l(k_val, k1, l, chi_grid, W_chi_array)
        I_k_k2[i] = calc_I_l(k_val, k2, l, chi_grid, W_chi_array)
    end
    
    # Moltiplichiamo direttamente per P_k_array senza chiamate a funzioni vettorizzate
    integrand_k = @. (k_grid^2) * P_k_array * I_k_k1 * I_k_k2
    
    return trapz(k_grid, integrand_k)
end


brute_force_S_lkk_gg (generic function with 1 method)

In [22]:
using NPZ

ℓ_range = LinRange(2, 200, 100)
xmin = 26
xmax = 7000
N = 2^15+1
x = LinRange(xmin, xmax, N)
kmin = 2.5 / xmax
kmax = 200 / 13

# Carichiamo i dati
W_chi_array = reverse(npzread("/Users/anvi/Desktop/cosmo/notebooks/out/galaxy_prefactor_W.npy"))
pk_array = npzread("/Users/anvi/Desktop/cosmo/notebooks/out/pk_grid.npy")
k_grid = Blast.get_clencurt_grid(kmin, kmax, 150)

# Preallochiamo e iteriamo sui l_val
S_res_array = zeros(length(ℓ_range))

for (i, l_val) in enumerate(ℓ_range)
    # kmin e kmax passati come k1 e k2
    S_res_array[i] = brute_force_S_lkk_gg(kmin, kmax, l_val, pk_array, x, W_chi_array, k_grid)
    println("Risultato brute force per l = ", l_val, ": ", S_res_array[i])
end


Risultato brute force per l = 2.0: 4.616526042756455e-11
Risultato brute force per l = 4.0: -2.038002087274909e-15
Risultato brute force per l = 6.0: 7.96499294331653e-18
Risultato brute force per l = 8.0: -2.12807076525318e-19
Risultato brute force per l = 10.0: -2.060081968663145e-21
Risultato brute force per l = 12.0: -1.201538165505638e-24
Risultato brute force per l = 14.0: -6.1354348168851384e-27
Risultato brute force per l = 15.999999999999998: -2.1898182908976406e-29
Risultato brute force per l = 18.0: -2.0617855405985297e-32
Risultato brute force per l = 20.0: -1.6793255937014752e-34
Risultato brute force per l = 22.0: -4.783945623401949e-37
Risultato brute force per l = 24.0: -1.2431926527301496e-40
Risultato brute force per l = 26.0: 3.3976077066276257e-43
Risultato brute force per l = 28.000000000000004: 1.7444263590015128e-46
Risultato brute force per l = 29.999999999999996: 1.4149251559453329e-49


LoadError: TaskFailedException

[91m    nested task error: [39mInterruptException:
    Stacktrace:
      [1] [0m[1mbesselj[22m[0m[1m([22m[90mnu[39m::[0mFloat64, [90mx[39m::[0mFloat64[0m[1m)[22m
    [90m    @[39m [32mSpecialFunctions[39m [90m~/.julia/packages/SpecialFunctions/AeoJp/src/[39m[90m[4mbessel.jl:521[24m[39m
      [2] [0m[1msphericalbesselj[22m
    [90m    @[39m [90m~/.julia/packages/SpecialFunctions/AeoJp/src/[39m[90m[4mbessel.jl:767[24m[39m[90m [inlined][39m
      [3] [0m[1mj_l[22m
    [90m    @[39m [90m./[39m[90m[4mIn[21]:4[24m[39m[90m [inlined][39m
      [4] [0m[1m_broadcast_getindex_evalf[22m
    [90m    @[39m [90m./[39m[90m[4mbroadcast.jl:709[24m[39m[90m [inlined][39m
      [5] [0m[1m_broadcast_getindex[22m
    [90m    @[39m [90m./[39m[90m[4mbroadcast.jl:682[24m[39m[90m [inlined][39m
      [6] [0m[1m_getindex[22m[90m (repeats 2 times)[39m
    [90m    @[39m [90m./[39m[90m[4mbroadcast

In [ ]:
include("blast_code/src/Blast.jl")
include("blast_code/src/blast_tutorials.jl")
using .Blast
using .blast_tutorials
include("blast_code/src/galaxy_galaxy.jl")
include("blast_code/src/shear_shear.jl")
using .galaxy_galaxy
using .shear_shear
include("blast_code/src/config.jl")
include("blast_code/src/paths.jl")
include("blast_code/src/plot_config.jl")

#### Defining an output folder for each run

#### Setting run parameters, plot parameters, and grids in k

In [ ]:
#sets the paths for the output directories 
paths = setup_output_directories()
#sets up the cosmology grid and returns the parameters
grid_data = setup_cosmology_grid() 
#sets up the plotting theme and returns the parameters
plot_theme = setup_plot_theme() 
#returns the k grids for the calculations
grids = Blast.generate_k_grids(grid_data.kmin, grid_data.kmax, grid_data.Nk, grid_data.Nkp, grid_data.Nkpp; sorting=false) 
# #saves the run parameters in a txt file
params_run = save_run_config(
    paths.output_dir, grid_data.N, 
    grid_data.xmin, grid_data.xmax, grid_data.zmin, grid_data.zmax, grid_data.kmin, grid_data.kmax, 
    grid_data.n_cheb, grid_data.ℓ, grid_data.Nk, grid_data.Nkp, grid_data.Nkpp,
    grid_data.x, grid_data.z, 
    grids.k_grid, grids.kp_grid, grids.kpp_grid, grids.sorting)
;

In [ ]:
println(grids.k_grid[1])
println(grids.k_grid[end])
println(grid_data.kmin)
println(grid_data.kmax)

### Galaxy clustering factor

$bias = b(z,z^2,z^3)$
comes from [this paper](https://arxiv.org/pdf/1807.10331)

Defining the kernel/window function for a galaxy probe: the window function is \
\
$W(z) = \frac{H(z) n(z) \chi(z)^2 b(z) D(z)}{c} $ \
\
where \
\
$n(z) = A (\frac{z}{z_0})^{\alpha} exp[{-(\frac{z}{z_0})^{\beta}}] $, \
\
with $A = \frac{1.5}{z_0}$, $\alpha = 2$ and $\beta = 1.5$ \
\
$b(z) = b_0 \sqrt{1+z} $ and $b_0 = 1$ \
\
$D(z) = \frac{D(z)^{unnorm}}{D(0)^{unnorm}}$, \
\
with $D(z)^{unnorm} = E(z) \int_z^{\infty} dz' \frac{1+z'}{E(z')^3} $ 

As for the ````gal_prefactor_W_cheb````, it is obtained interpolating each factor, ````bias````, ````growth````, ````nz_norm````, ````chi```` on the ````z````, and then obtaining the total interpolated product ````gal_prefact_W_cheb````.

As for the ````cheb_coeff_gal````, I compute this similarly to Blast: I define a ````plan```` object that takes the ````gal_prefact_W_cheb```` as input, and then returns the ````cheb_coeff```` as the output of the functions ````fast_chebcoefs````

In [ ]:
gal_prefact_W, bias, growth, Hubble_param, nz_norm = galaxy_galaxy.galaxy_prefactor(grid_data.x, grid_data.z, grid_data.cosmo; 
                                                     output_dir=paths.output_dir, plot_style = plot_theme.shared_style);
gal_prefact_W_cheb = galaxy_galaxy.galaxy_prefactor_cheb(grid_data.xmin, grid_data.xmax, grid_data.n_cheb, grid_data.z, grid_data.x, 
                                                         bias, growth, Hubble_param, nz_norm; 
                                                         output_dir=paths.output_dir);

Is it true that

$W(\chi) \approx \sum_{n=0}^{N_{cheb}-1} c_n T_n(\chi)$ ?

It seems that the kernel computed on the grid of comoving distances $\chi$, when computed on the Chebyshev nodes, has this behaviour

In [ ]:
println("Kernel on normal nodes at the end of the array: ")
println(gal_prefact_W[end])
println("Kernel on Chebyshev nodes at the start of the array: ")
println(gal_prefact_W_cheb[1])
println("Kernel on normal nodes at the start of the array: ")
println(gal_prefact_W[1])
println("Kernel on Chebyshev nodes at the end of the array: ")
println(gal_prefact_W_cheb[end])

So one could verify whether the two object are the same just by indexing in a reverse way the Chebyshev object

In [ ]:
# questo diventa superfluo.
gal_prefact_cheb_ord = reverse!(gal_prefact_W_cheb); #this is the same as doing gal_prefact_cheb_ord = gal_prefact_W_cheb[end:-1:1];

In [ ]:
cheb_coeff_gal = galaxy_galaxy.compute_prefactor_chebcoeffs(gal_prefact_cheb_ord; output_dir=paths.output_dir);

In [ ]:
println("Kernel on normal nodes at the end of the array: ")
println(gal_prefact_W[end])
println("Kernel on Chebyshev nodes at the start of the array, now reversed: ")
println(gal_prefact_cheb_ord[end])

Here I plot che object on normal nodes vs the N comoving distances $\chi$, and the object on Chebyshev nodes vs the $n_{cheb}$ comoving distances

In [ ]:
x_cheb_plot = Blast.get_clencurt_grid(grid_data.xmax, grid_data.xmin, grid_data.n_cheb)
z_interp = DataInterpolations.AkimaInterpolation(grid_data.z, grid_data.x, extrapolation=ExtrapolationType.Linear)
z_cheb_plot = z_interp.(x_cheb_plot)
plot(x_cheb_plot, gal_prefact_cheb_ord, label = L"\sum_{n=0}^{N_{cheb}-1} c_n T_n(\chi)", ls=:dash, markersize=0.1, markercolor =:blue)
plot!(x_cheb_plot, gal_prefact_W_cheb, label = L"W(\chi) \; \mathrm{on} \; \mathrm{Chebyshev} \; \mathrm{nodes}", lw=1, ls=:dot; plot_theme.shared_style...)
plot!(grid_data.x, gal_prefact_W, label = L"W(\chi)", lw=1, ls=:dot; plot_theme.shared_style...)

Now I want to compute the relative error between the $W(z)$ and $W(z_cheb)$. To do so, they have to share the same grid. So I bring the normal W(z), previously computed on N nodes, on the Chebyshev grid. 
Note that the object is computed using both $\chi$ and $z$ obtained through get_clencurt_grid, so there's no need to reverse it at the end. 

In [ ]:
galW_computed_on_cheb, _, _, _, _ = galaxy_galaxy.galaxy_prefactor(x_cheb_plot, z_cheb_plot, grid_data.cosmo; output_dir=paths.output_dir);

Now that I have the two object on the same grid, I once again plot the two

In [ ]:
plot(x_cheb_plot, gal_prefact_W_cheb, label = L"\sum_{n=0}^{N_{cheb}-1} c_n T_n(\chi)", ls=:dash, markersize=0.1, markercolor =:blue)
plot!(x_cheb_plot, galW_computed_on_cheb, label = L"W(\chi))", lw=1, ls=:dot)
plot!(dpi = 300, legendposition = :topleft, xlabel = L"\chi [Mpc/h]", ylabel = L"W(\chi) [\mathrm{Mpc}/h]",
        title = "Chebyshev approximation of the galaxy prefactor W"; plot_theme.shared_style...)

In [ ]:
rel_err = (gal_prefact_W_cheb ./ galW_computed_on_cheb) .- 1
rel_err_abs = abs.(rel_err)
rel_err_pct = 100 .* rel_err
rel_err_pct_abs = 100 .* rel_err_abs
;

In [ ]:
plot(
    x_cheb_plot, rel_err_pct,
    label = L"\frac{W_{\mathrm{Cheb}} - W_{\mathrm{true}}}{W_{\mathrm{true}}}\,[\%]",
    lw = 1.5,
    marker = :circle,
    markersize = 2,
    legendposition = :topright,
    xlabel = L"\chi\,[\mathrm{Mpc}/h]",
    ylabel = L"\mathrm{errore\ relativo}\,[\%]",
    title = "Relative error of Chebyshev approximation", size=plot_theme.size_Cl; plot_theme.shared_style...
    )

hline!([0.0], ls = :dash, lw = 1, color = :black, label = false)


In [ ]:
p1 = plot(
    x_cheb_plot, galW_computed_on_cheb,
    label = L"\sum_{n=0}^{N_{\mathrm{cheb}}-1} c_n T_n(\chi)",
    ls = :dash, color = :green)
plot!(p1, x_cheb_plot, gal_prefact_W_cheb, label = L"W(\chi)", lw = 1, ls = :dot, color = :red)
plot!(p1, dpi = 300, legendposition = :topleft, xlabel = L"\chi [Mpc/h]", ylabel = L"W(\chi) [\mathrm{Mpc}/h]", 
        size = plot_theme.size_Cl; plot_theme.shared_style...
        )
    
p2 = plot(
    x_cheb_plot, rel_err_pct,
    label = L"\frac{W_{\mathrm{Cheb}} - W_{\mathrm{true}}}{W_{\mathrm{true}}} [\%]",
    lw = 1.0, marker = :circle, markercolor = :black, markersize = 1, linestyle = :dot, 
    title = "relative error [%]", titlefontsize = 10
)
hline!(p2, [0.0], ls = :solid, lw = 1, color = :red, label = false)

plot(p1, p2, layout = @layout([a; b]), dpi = 300,
     xlabel = L"\chi\,[\mathrm{Mpc}/h]",
    size = plot_theme.size_Cl; plot_theme.shared_style...)

### The dimension of the Chebyshev coefficients is [n_cheb]

In [ ]:
println("size of cheb_coeff_gal: ", size(cheb_coeff_gal))
npzwrite(joinpath(paths.quantity_subdir, "chebcoefs/cheb_coeff_gal.npy"), cheb_coeff_gal)

In [ ]:
n_idx = 0:(grid_data.n_cheb - 1)
plot(n_idx, abs.(cheb_coeff_gal),
     yscale = :log10,
     xlabel = L"{grid_data.n_cheb}",
     ylabel = L"|c_{grid_data.n_cheb}|",
     label = L"|c_{grid_data.n_cheb}| \ \mathrm{di} \ W(\chi)",
     title = "Chebyshev coefficients decay",
     marker = :circle, markersize = 3; plot_theme.shared_style...
     )
savefig(joinpath(paths.plot_subdir, "kernels", "chebcoeff_decay_galaxy.png"))

In [ ]:
function cheb_eval_partial(coeffs, x, xmin, xmax, ntrunc)
    t = @. (2*x - (xmax + xmin)) / (xmax - xmin)
    T0 = ones(length(x))
    if ntrunc == 0
        return coeffs[1] .* T0
    end
    T1 = t
    S = coeffs[1] .* T0 .+ coeffs[2] .* T1
    for n in 2:ntrunc
        T2 = @. 2*t*T1 - T0
        S .+= coeffs[n+1] .* T2
        T0, T1 = T1, T2
    end
    return S
end

chi_cheb_nodes = Blast.get_clencurt_grid(grid_data.xmin, grid_data.xmax, grid_data.n_cheb)
x_ref = chi_cheb_nodes
Wref = gal_prefact_W_cheb

truncs = 1:1:length(cheb_coeff_gal)-1
#truncs = [5, 10, 20, 40, 80, 120, 160, length(cheb_coeff_gal)-1]
errs = Float64[]
Wn = zeros(length(x_ref))
for ntrunc in truncs
    Wn = cheb_eval_partial(cheb_coeff_gal, x_ref, grid_data.xmin, grid_data.xmax, ntrunc)
    err = maximum(abs.(Wn .- Wref)) / maximum(abs.(Wref))
    push!(errs, err)
end

truncs_plot = truncs[1:end]
errs_plot = errs[1:end]
yticks_vals = 10.0 .^ (floor(log10(minimum(errs_plot))):ceil(log10(maximum(errs_plot))))

plot(truncs_plot, errs_plot,
     yscale = :log10,
     marker = :circle,
     xlabel = L"N_{trunc}",
     ylabel = L"\mathrm{err}(N_{trunc}) = \frac{\max_i |W_{N_{trunc}} - W(\chi_i)|}{\max_i |W(\chi_i)|}",
     title  = L"\mathrm{err}(N_{trunc}) = \frac{\max_i |W_{N_{trunc}} - W(\chi_i)|}{\max_i |W(\chi_i)|}",
     legend = false,
     framestyle = :box,
     yticks = yticks_vals, size=plot_theme.size_Cl; plot_theme.shared_style...
     )
savefig(joinpath(paths.plot_subdir, "kernels", "chebcoeff_truncation_error_galaxy.png"))

$W_{tilde} = \int dz W(z) j_l(k\chi(z)) j_l(k_1\chi(z))$

$\tilde W_{\ell}^g(k1,k) \approx \sum_{n=0}^{N_{cheb}-1} c_n \int_{z_{min}}^{z_{max}} dz T_n(\hat z) k_1 j_l(k\chi(z)) j_l(k_1\chi(z))$

### The dimension of the $\tilde W$ matrix is [$N_k$, $N_{kp}$, $n_{cheb}$, $\ell$]

In [ ]:
# # N = 2^15+1, Nk = Nkp = Nkpp = 150, N_cheb = 200, length(ℓ) = 100
reuse = true
mode = "fast"
W_tilde = zeros(grid_data.Nk, grid_data.Nkp, grid_data.n_cheb, length(grid_data.ℓ))
if reuse
  if mode == "fast"
    W_tilde = npzread("/Users/anvi/Desktop/cosmo/notebooks/out/W_tilde_fast.npy")
  elseif mode == "slow"
    W_tilde = npzread("/Users/anvi/Desktop/cosmo/notebooks/out/W_tilde_slow.npy")
  end
else
  elapsed_time = zeros(length(grid_data.ℓ))
  println("Dimensions of W_tilde: ", size(W_tilde))
  for i in eachindex(grid_data.ℓ)
      t_0 = time()
      W_tilde[:, :, :, i] .= Blast.W_tilde_computation(grid_data.ℓ[i], grid_data.zmin, grid_data.zmax, grid_data.kmin, grid_data.kmax,
                                                   grid_data.Nk, grid_data.Nkp, grid_data.n_cheb, grid_data.N, grids.k_grid, grids.kp_grid, grids.kpp_grid)
      t_end = time()
      elapsed_time[i] = t_end - t_0
      println("Finished computing W_tilde for ℓ = $(grid_data.ℓ[i]). Time elapsed $(round(elapsed_time[i], digits=2))s")
  npzwrite(joinpath(paths.quantity_subdir, "elapsed_time.npy"), elapsed_time)  
  histogram(grid_data.ℓ, elapsed_time, bins=30, xlabel = L"\ell", ylabel = "Elapsed time [s]", title = "Elapsed time for W_tilde computation", size=plot_theme.size_plot; plot_theme.shared_style...)  
  end
end
;

In [ ]:
#npzwrite(joinpath(paths.output_dir, "W_tilde.npy"), W_tilde)

In [ ]:
#isapprox(npzread("/Users/anvi/Desktop/cosmo/notebooks/out/W_tilde_fast.npy"), npzread("/Users/anvi/Desktop/cosmo/notebooks/out/W_tilde_slow.npy"), atol=1e-10)
#true

In [ ]:
println("Size of W_tilde: ", size(W_tilde))
#Size of W_tilde: (Nk, Nkp, Ncheb, Nl)

In [ ]:
# println("min = ", minimum(W_tilde), ", max = ", maximum(W_tilde))
# println("mean = ", mean(W_tilde), ", std = ", std(W_tilde))
# if  any(isnan, W_tilde)
#     println("There are NaN values in W_tilde")
# end
# if any(isinf, W_tilde)
#     println("There are infinite values in W_tilde")
# end
# if any(iszero, W_tilde)
#     println("There are zero values in W_tilde")
# end

$\tilde W(k,k_1)$ (like $\tilde W(k,k_2)$) represents the term: \
$\tilde W_{i,p,l}^{(\ell)} = \sum_{m=1}^{N_k} w_{k_m} T_{\ell}(k_m) j_{\ell}(\chi_i k_m) j_{\ell}(\chi_p k_m) $ \
it describes how much two shells at comoving distance $\chi_i$ and $\chi_p$ are correlated to the multipole $\ell$, weighted by the Chebyshev polynomial $T_{\ell}$ on the mode $k$.

### The dimension of $W_{final}^{gal}$ is [$\ell$, $N_k$, $N_{kp}$]

In [ ]:
@tullio W_final_gal[il, ik, ikp] := W_tilde[ik, ikp, ic, il] * cheb_coeff_gal[ic]
;

In [ ]:
println("Size of W_final_gal: ", size(W_final_gal))
# [ell, k_grid, kp_grid]

In [ ]:
# println("min = ", minimum(W_final_gal), ", max = ", maximum(W_final_gal))
# println("mean = ", mean(W_final_gal), ", std = ", std(W_final_gal))
# if  any(isnan, W_final_gal)   
#     println("There are NaN values in W_final_gal")
# end
# if any(isinf, W_final_gal)   
#     println("There are infinite values in W_final_gal")
# end
# if any(iszero, W_final_gal)
#     println("There are zero values in W_final_gal")
# end

### $k_{grid}$

In [ ]:
idx = sortperm(grids.k_grid)
idx_p = sortperm(grids.kp_grid)

# genera tick automatici alle potenze di 10 nel range dei dati
xticks_vals = 10.0 .^ (floor(log10(minimum(grids.k_grid))):ceil(log10(maximum(grids.k_grid))))
yticks_vals = 10.0 .^ (floor(log10(minimum(grids.kp_grid))):ceil(log10(maximum(grids.kp_grid))))

heatmap(grids.k_grid[idx], grids.kp_grid[idx_p],
        W_final_gal[1,idx,idx_p]/maximum(W_final_gal[1,idx,idx_p]),
        title=L"W_{final}^{gg}"*"(at fixed "*L"ℓ)",
        xscale = :log10, yscale = :log10,
        xlabel=L"k", ylabel=L"k_p",
        size = plot_theme.size_heatmap,
        c = plot_theme.c, 
        xticks = xticks_vals, yticks = yticks_vals; plot_theme.shared_style...
        )

In [ ]:
savefig(joinpath(paths.plot_subdir, "Sl_plots/k_grid_sorted_k_grid_sorted.png"))

In [ ]:
println(size(W_tilde))

In [ ]:
heatmap(1:grid_data.Nk, 1:grid_data.Nkp, W_final_gal[10,:,:]/maximum(W_final_gal[10,:,:]), 
    title = L"W_{final}^{gg}"*"(at fixed "*L"ℓ)", 
    xlabel = L"N_k", ylabel = L"N_{kp}", 
    colorbar = true, c = plot_theme.c, size = plot_theme.size_heatmap; plot_theme.shared_style...
    )


In [ ]:
savefig(joinpath(paths.plot_subdir, "Sl_plots/Nk_Nkp.png"))

In [ ]:
idx_p = sortperm(grids.kp_grid)
yticks_vals = 10.0 .^ (floor(log10(minimum(grids.k_grid))):ceil(log10(maximum(grids.k_grid))))
heatmap(1:grid_data.Nk, 
        grids.kp_grid[idx_p], 
        W_final_gal[90, :, idx_p] / maximum(W_final_gal[90, :, idx_p]), 
        title = L"W_{final}^{gg}"*L"("*"at fixed "*L"ℓ = 90)", 
        xlabel = L"N_k", ylabel = L"\log_{10}(k_{grid})",
        yscale = :log10, colorbar = true, c = plot_theme.c, size = plot_theme.size_heatmap; plot_theme.shared_style...
        )


In [ ]:
savefig(joinpath(paths.plot_subdir, "Sl_plots/Nk_kp_grid_sorted.png"))

In [ ]:
#3D matter power spectrum
pk_dict = npzread("blast_code/data/pk.npz")
Pklin = pk_dict["pk_lin"]
Pknonlin = pk_dict["pk_nl"]
k_pk = pk_dict["k"]
z_pk = pk_dict["z"]
#Interpolating the power spectrum: Linear P(k) - Non-linear P(k)
y_pk = LinRange(log10(first(k_pk)),log10(last(k_pk)), length(k_pk))
x_pk = LinRange(first(z_pk), last(z_pk), length(z_pk))
InterpPmm = Interpolations.interpolate(log10.(Pklin),BSpline(Cubic(Line(OnGrid()))))
InterpPmm = scale(InterpPmm, (x_pk, y_pk))
InterpPmm = Interpolations.extrapolate(InterpPmm, Line())
InterpPmm_nl = Interpolations.interpolate(log10.(Pknonlin),BSpline(Cubic(Line(OnGrid()))))
InterpPmm_nl = scale(InterpPmm_nl, x_pk, y_pk)
InterpPmm_nl = Interpolations.extrapolate(InterpPmm_nl, Line())
power_spectrum(k_pk, χ1, χ2) = @. sqrt(10^InterpPmm(grid_data.z_of_χ(χ1),log10(k_pk)) * 10^InterpPmm(grid_data.z_of_χ(χ2),log10(k_pk)))
power_spectrum_nl(k_pk, χ1, χ2) = @. sqrt(10^InterpPmm_nl(grid_data.z_of_χ(χ1),log10(k_pk)) * 10^InterpPmm_nl(grid_data.z_of_χ(χ2),log10(k_pk)))

In [ ]:
plot(k_pk, power_spectrum.(k_pk, 1000.0, 1000.0), 
     label="Linear P(k)", 
     xscale=:log10, 
     yscale=:log10, 
     title=L"$P(k)$ at $z=0$", titlefontsize=20,
     xlabel=L"$k \; (h/\mathrm{Mpc})$", 
     ylabel=L"$P(k) \; ((\mathrm{Mpc}/h)^3)$", 
     labelfontsize=15)
plot!(k_pk, power_spectrum_nl.(k_pk, 1000.0, 1000.0), 
      label="Non-linear P(k)", 
      xscale=:log10, 
      yscale=:log10, size=plot_theme.size_Cl; plot_theme.shared_style...
      )

get_clencurt_grid produces the node of Clenshaw-Curtis mapped on [$k_{min}$, $k_{max}$]. \
get_clencurt_weights produces the corresponding quadrature weights scaled to the interval [-1,1]. \

In [ ]:
idx = sortperm(grids.k_grid)
idx_p = sortperm(grids.kp_grid)
;

In [ ]:
Pk_grid = power_spectrum.(grids.k_grid, grid_data.kmin, grid_data.kmax)
w_k = Blast.get_clencurt_weights(grid_data.kmin, grid_data.kmax, grid_data.Nk)
weight_gal = w_k .* grids.k_grid.^2 .* Pk_grid 
;

In [ ]:
plot(grids.k_grid[idx], Pk_grid[idx], 
     xscale=:log10, 
     yscale=:log10, 
     title=L"$P(k)$", titlefontsize=20,
     xlabel=L"$k \; (h/\mathrm{Mpc})$", 
     ylabel=L"$P(k) \; ((\mathrm{Mpc}/h)^3)$", 
     labelfontsize=15, size=plot_theme.size_plot; plot_theme.shared_style...
     )

In [ ]:
plot(grids.k_grid, w_k, label = L"weights", xscale = :log10, 
     yscale = :log10, 
     xlabel = L"k \; (h/\mathrm{Mpc})", size=plot_theme.size_Cl; plot_theme.shared_style...
     )
plot!(grids.k_grid, grids.k_grid.^2, label = L"k^2", xscale = :log10, 
     yscale = :log10, 
     xlabel = L"k \; (h/\mathrm{Mpc})", size=plot_theme.size_Cl; plot_theme.shared_style...
     )
plot!(grids.k_grid, Pk_grid, label = L"P(k)", xscale = :log10, 
     yscale = :log10, 
     xlabel = L"k \; (h/\mathrm{Mpc})", size=plot_theme.size_Cl; plot_theme.shared_style...
     )
plot!(grids.k_grid, weight_gal, 
     xscale=:log10, 
     yscale=:log10, 
     title=L"$k^2 P(k)$", titlefontsize=20,
     xlabel=L"$k \; (h/\mathrm{Mpc})$", 
     ylabel=L"$k^2 P(k) \; ((\mathrm{Mpc}/h))$", 
     label =L"$k^2 P(k) \; * \; \mathrm{w} $",
     labelfontsize=15, legendposition = :bottomright, size=plot_theme.size_plot; plot_theme.shared_style...
     )
plot!(grids.k_grid[idx], weight_gal[idx], 
     xscale=:log10, 
     yscale=:log10, 
     title=L"$k^2 P(k)$", titlefontsize=20,
     xlabel=L"$k \; (h/\mathrm{Mpc})$", 
     ylabel=L"$k^2 P(k) \; ((\mathrm{Mpc}/h))$", 
     label =L"$k^2 P(k) \; * \; \mathrm{w} \; \mathrm{ordered}$",
     labelfontsize=15, legendposition = :bottomright, size=plot_theme.size_plot; plot_theme.shared_style...
     )

In [ ]:
plot(grids.k_grid[idx], weight_gal[idx], 
     xscale=:log10, 
     yscale=:log10, 
     title=L"$k^2 P(k)$", titlefontsize=20,
     xlabel=L"$k \; (h/\mathrm{Mpc})$", 
     ylabel=L"$k^2 P(k) \; ((\mathrm{Mpc}/h))$", 
     label =L"$k^2 P(k) \; * \; \mathrm{w} $",
     labelfontsize=15, legendposition = :bottomright, size=plot_theme.size_plot; plot_theme.shared_style...
     )

In [ ]:
abstract type AbstractProbe end
struct Galaxy <: AbstractProbe end
struct Shear <: AbstractProbe end
factorial_frac(ℓ) = (ℓ + 2.0) * (ℓ + 1.0) * ℓ * (ℓ - 1.0)
get_ell_prefactor(::Galaxy, ::Galaxy, ℓ) = @. (2 / π) * ones(length(ℓ))
get_ell_prefactor(::Galaxy, ::Shear,  ℓ) = @. (2 / π) * sqrt(factorial_frac(ℓ))
get_ell_prefactor(::Shear,  ::Galaxy, ℓ) = @. (2 / π) * sqrt(factorial_frac(ℓ))
get_ell_prefactor(::Shear,  ::Shear,  ℓ) = @. (2 / π) * factorial_frac(ℓ)
pref_gg = get_ell_prefactor(Galaxy(), Galaxy(), grid_data.ℓ)
pref_gg = reduce(vcat, pref_gg)
pref_gs = get_ell_prefactor(Galaxy(), Shear(), grid_data.ℓ)
pref_gs = reduce(vcat, pref_gs)
pref_gg = reduce(vcat, pref_gg)
pref_ss = get_ell_prefactor(Shear(), Shear(), grid_data.ℓ)
pref_ss = reduce(vcat, pref_ss);

In [ ]:
println("SIZES")
println("weight_gal -> ", size(weight_gal))
println("W_final_gal -> ", size(W_final_gal))
println("pref_gg -> ", size(pref_gg))

In [ ]:
S_lkk_gg = zeros(Float64, size(W_final_gal, 3), size(W_final_gal, 3), length(grid_data.ℓ))
@tullio S_lkk_gg[kp, kpp, li] = pref_gg[li] * weight_gal[k] * W_final_gal[li, k, kp] * W_final_gal[li, k, kpp]
npzwrite(joinpath(paths.quantity_subdir, "Sl/S_lkk_gg.npy"), S_lkk_gg)
println("Size of S_l (kp, kpp) (gal-gal): \n(grid_data.Nk, grid_data.Nkp, NL) -> ", size(S_lkk_gg))
;

wavenumber k as a function of $\ell$ through $k \approx \frac{\ell+0.5}{\chi}$ ?

In [ ]:
#S_lkk_gg_ordered = reverse(S_lkk_gg, dims=(1, 2))

In [ ]:
npzwrite(joinpath(paths.quantity_subdir, "Sl/S_lkk_gg.npy"), S_lkk_gg)

In [ ]:
xref = (grid_data.xmax - grid_data.xmin ) * 0.5
ℓ_to_k = ℓ -> ℓ ./ xref
ℓ_ticks = 1:50:200
k_ticks = ℓ_to_k.(ℓ_ticks)
k_ticklabels = [string(round(k, digits=3)) for k in k_ticks];

In [ ]:
i = 1
j = i 
plot(grid_data.ℓ, S_lkk_gg[i,j,:],
      color = :blue,
      label = L"i = $i, j = $j",)
      
plot!(xaxis = L"\ell",
      ylabel = L"S_\ell",
      xscale = :log10,
     )
plot!(twiny(),
      xaxis = L"k \; (h/\mathrm{Mpc})",
      xticks = (ℓ_ticks, k_ticklabels),
      xscale = :log10,
      xlims = (minimum(grid_data.ℓ), maximum(grid_data.ℓ)),
      label = ""
     )
plot!(label = "Beyond BLAST", 
      size=plot_theme.size_Cl,
      title = L"S_\ell^{gg} (i_k = i_{k_p} = %$i)",
      titlefontsize = 20,
      titleposition = :left ; plot_theme.shared_style...)

In [ ]:
plt = plot()
idx = sortperm(grids.k_grid)
il = 1
ikp = 145
k_p = grids.kp_grid[ikp]
plot(grids.k_grid[idx], S_lkk_gg[idx,idx,il], color = plot_theme.colors[idx], linestyle = :solid)
plot!(xaxis = L"k_{grid} \; (h/\mathrm{Mpc})",
      ylabel = L"S_\ell"
      ,xscale = :log10, label = ""
     )
plot!( title = L"S_\ell^{gg} = \int dk k^2 P(k) \int \tilde W(k, k_1) \int \tilde W(k, k_2) -> (\mathrm{varying} \; k, \mathrm{at} \; \mathrm{different} \; k_p)",
      titlefontsize = 20,
      label = L"\ell=%$(grid_data.ℓ[il]), k_p=%$(round(k_p, digits=3)) \; \mathrm{h/Mpc}",
      titleposition = :left, legend = nothing,
      #legendposition = :outertopright, 
      size=plot_theme.size_Cl; plot_theme.shared_style...)

In [ ]:
plt = plot()
idx = sortperm(grids.k_grid)
il = 1
ikp = 1
k_p = grids.kp_grid[ikp]
plot(grids.k_grid[idx], S_lkk_gg[idx,il,ikp],
      color = :blue, linestyle = :dash,
      label = L"\ell=%$(grid_data.ℓ[il]), k_p = %$(round(k_p, digits=3))")
il = 10
plot!(grids.k_grid[idx], S_lkk_gg[idx,il,ikp],
      color = :red,
      label = L"\ell=%$(grid_data.ℓ[il]), k_p = %$(round(k_p, digits=3))")
il = 50
plot!(grids.k_grid[idx], S_lkk_gg[idx,il,ikp],
      color = :green,
      label = L"\ell=%$(grid_data.ℓ[il]), k_p = %$(round(k_p, digits=3))")
il = 75
plot!(grids.k_grid[idx], S_lkk_gg[idx,il,ikp],
      color = :purple,
      label = L"\ell=%$(grid_data.ℓ[il]), k_p = %$(round(k_p, digits=3))")
il = 100
plot!(grids.k_grid[idx], S_lkk_gg[idx,il,ikp],
      color = :black, linestyle = :dot,
      label = L"\ell=%$(grid_data.ℓ[il]), k_p = %$(round(k_p, digits=3))")
plot!(xaxis = L"k_{grid} \; (h/\mathrm{Mpc})",
      ylabel = L"S_\ell"
      #,xscale = :log10
     )
plot!(label = "Beyond BLAST", 
      title = L"S_\ell^{gg} (k \; \mathrm{varies})",
      titlefontsize = 20,
      titleposition = :left,
      legendposition = :topright, size=plot_theme.size_Cl; plot_theme.shared_style...)

In [ ]:
plt = plot()
kp = 120
for il in eachindex(grid_data.ℓ)
          plot!(grids.k_grid, S_lkk_gg[:,kp,il],
          color = plot_theme.colors[il]
          ,label = L"\ell = %$(round(grid_data.ℓ[il], digits=2))"
          )
end 

plot!(xaxis = L"k_{grid} \; (h/\mathrm{Mpc})",
      ylabel = L"S_\ell"
      #,xscale = :log10,
     )
plot!(label = "Beyond BLAST", 
      title = L"S_\ell^{gg} (k \; \mathrm{varies}, k_p = %$kp, \ell \; \mathrm{varies})",
      titlefontsize = 20,
      titleposition = :left,
      legendposition = :outertopright, size=plot_theme.size_Cl; plot_theme.shared_style...
)
plt

In [ ]:
plt = plot()

for i in 1:grid_data.Nkp
    plot!(plt, grid_data.ℓ, S_lkk_gg[i,i,:], line_z = i,
          label = L"i = %$(i)",
          color = plot_theme.colors, linewidth = 2)
end

plot!(plt, label="Beyond BLAST", 
     xaxis=L"\ell", ylabel=L"S_\ell", 
     xscale = :log10
    )
plot!(twiny(),
      xaxis = L"k \; (h/\mathrm{Mpc})",
      xticks = (ℓ_ticks, k_ticklabels),
      xlims = (minimum(grid_data.ℓ), maximum(grid_data.ℓ)),
      xscale = :log10,
      label = ""
     )
plot!(legend = false, colorbar = true, colorbar_title = L"i = j",
      clims = (1, grid_data.Nkp),
      titleposition = :left,
      title=L"S_\ell^{gg} (i = j)", size=plot_theme.size_Cl; plot_theme.shared_style...
     )
plt

In [ ]:
plt = plot()

for i in 1:grid_data.Nk
    plot!(plt, grid_data.ℓ, S_lkk_gg[i,i,:]/maximum(S_lkk_gg[i,i,:]), line_z = i,
          label = L"i = %$(i)",
          color = plot_theme.colors, linewidth = 1)
end

plot!(plt, 
    clims = (1, grid_data.Nk),
    xaxis=L"\ell", yaxis=L"S_\ell", 
    xscale = :log10
    )

plot!(twiny(),
      xaxis = L"k \; (h/\mathrm{Mpc})",
      xticks = (ℓ_ticks, k_ticklabels),
      xlims = (minimum(grid_data.ℓ), maximum(grid_data.ℓ)),
      label = ""
     )

plot!(label = "Beyond BLAST", 
    legend = false, colorbar = true, colorbar_title = L"j",
    title=L"S_\ell^{gg} (i = j)",
    size=plot_theme.size_Cl, titlefontsize=20, 
    titleposition = :left; plot_theme.shared_style...
    )

plt

In [ ]:
savefig(joinpath(paths.plot_subdir, "Sl_plots/S_lkk_gg_fixed_l_various.png"))

In [ ]:
plt = plot()

a = 1:5:grid_data.Nk
for i in a
    plot!(plt, grid_data.ℓ, S_lkk_gg[i,i,:]/maximum(S_lkk_gg[i,i,:]), line_z = i,
          label = L"i = %$(i)", color = plot_theme.colors, linewidth = 1
          )
end

plot!(plt, 
    xaxis=L"\ell", yaxis=L"S_\ell", 
    xscale = :log10
    )

plot!(twiny(),
      xaxis = L"k \; (h/\mathrm{Mpc})",
      xticks = (ℓ_ticks, k_ticklabels),
      xlims = (minimum(grid_data.ℓ), maximum(grid_data.ℓ))     
      )

plot!(label = "Beyond BLAST", 
    legend = false, colorbar = true, colorbar_title = L"i_k", clims = (1, grid_data.Nkp),
    title=L"S_\ell^{gg} (i = j)",
    size=plot_theme.size_Cl, titlefontsize=20, 
    titleposition = :left; plot_theme.shared_style...
    )

plt

In [ ]:
plt = plot()
i_fixed = 1
for j in 1:grid_data.Nkp
    plot!(plt, grid_data.ℓ, S_lkk_gg[i_fixed, j, :],
          line_z = j, color = plot_theme.colors, linewidth = 2
        )                  
end

plot!(plt,
    xaxis = L"\ell", yaxis = L"S_\ell",
    xscale = :log10
    )

plot!(twiny(),
      xaxis = L"k \; (h/\mathrm{Mpc})",
      xticks = (ℓ_ticks, k_ticklabels),
      xlims = (minimum(grid_data.ℓ), maximum(grid_data.ℓ))
     )

plot!(label = "Beyond BLAST", 
    legend = false, colorbar = true, colorbar_title = L"j", clims = (1, grid_data.Nkp),
    title = L"S_\ell^{gg} (i = %$i_fixed)",
    size=plot_theme.size_Cl, titlefontsize=20, 
    titleposition = :left;
    plot_theme.shared_style...
    )

plt


In [ ]:
plt = plot()
jj = 96
i_fixed = 1
for j in 1:jj
    plot!(plt, grid_data.ℓ, S_lkk_gg[i_fixed, j, :] / maximum(S_lkk_gg[i_fixed, j, :]),
          line_z = j, color = plot_theme.colors, linewidth = 2
          )
end

plot!(plt,
    xaxis = L"\ell", yaxis = L"S_\ell",
    xscale = :log10
    )

plot!(twiny(),
      xaxis = L"k \; (h/\mathrm{Mpc})",
      xticks = (ℓ_ticks, k_ticklabels),
      xlims = (minimum(grid_data.ℓ), maximum(grid_data.ℓ))
     )

plot!(label = "Beyond BLAST", 
    legend = false, colorbar = true, colorbar_title = L"j", clims = (1, grid_data.Nkp),
    title = L"S_\ell^{gg} (i = %$i_fixed,\ j)",
    size=plot_theme.size_Cl, titlefontsize=20, 
    titleposition = :left;
    plot_theme.shared_style...
    )

plt

In [ ]:
plt = plot()
jj = 100
i_fixed = 50
for j in 1:jj
    plot!(plt, grid_data.ℓ, S_lkk_gg[i_fixed, j, :],
          line_z = j, color = plot_theme.colors,
          linewidth = 2
          )
end

plot!(plt,
    xaxis = L"\ell", yaxis = L"S_\ell",
    xscale = :log10
    )

plot!(twiny(),
      xaxis = L"k \; (h/\mathrm{Mpc})",
      xticks = (ℓ_ticks, k_ticklabels),
      xlims = (minimum(grid_data.ℓ), maximum(grid_data.ℓ))
    )

plot!(label = "Beyond BLAST", 
    legend = false, colorbar = true, colorbar_title = L"j", clims = (1, grid_data.Nkp),
    title = L"S_\ell^{gg} (i = %$i_fixed,\ j)",
    size=plot_theme.size_Cl, titlefontsize=20, 
    titleposition = :left;
    plot_theme.shared_style...
    )

plt

In [ ]:
plt = plot()
k1 = 1
k2 = 1
plot(
    grid_data.ℓ,
    S_lkk_gg[1, 1, :] .* grid_data.ℓ .* (grid_data.ℓ .+ 1),
    xaxis = L"\ell",
    yaxis = L"\ell(\ell+1)S_\ell",
)

plot!(twiny(),
      xaxis = L"k \; (h/\mathrm{Mpc})",
      xticks = (ℓ_ticks, k_ticklabels),
      xlims = (minimum(grid_data.ℓ), maximum(grid_data.ℓ)),
      label = ""
     )

plot!(label = "Beyond BLAST", 
    legend = false, colorbar = true, colorbar_title = L"j", clims = (1, grid_data.Nkp),
    title = L"\ell(\ell+1)S_\ell^{gg}(k_1=%$k1,k_2=%$k2)",
    size=plot_theme.size_Cl, titlefontsize=20, 
    titleposition = :left;
    plot_theme.shared_style...
    )

In [ ]:
plt = plot()

for j in 1:grid_data.Nkp
    plot!(plt, grid_data.ℓ, S_lkk_gg[j, j, :] .* grid_data.ℓ .* (grid_data.ℓ .+ 1),
          line_z = j, color = plot_theme.colors, linewidth = 2
          )
end
    
plot!(plt,
    xaxis = L"\ell", 
    yaxis = L"\ell(\ell+1)S_\ell",
    xscale = :log10
    )

plot!(twiny(),
      xaxis = L"k \; (h/\mathrm{Mpc})",
      xticks = (ℓ_ticks, k_ticklabels),
      xlims = (minimum(grid_data.ℓ), maximum(grid_data.ℓ))
     )

plot!(label = "Beyond BLAST", 
    legend = false, colorbar = true, colorbar_title = L"j", clims = (1, grid_data.Nkp),
    title = L"\ell(\ell+1)S_\ell^{gg}(k_1,k_2)",
    size=plot_theme.size_Cl, titlefontsize=20, 
    titleposition = :left;
    plot_theme.shared_style...
    )

plt

In [ ]:
plt = plot()

for j in a
    plot!(plt, grid_data.ℓ, S_lkk_gg[j, j, :] .* grid_data.ℓ .* (grid_data.ℓ .+ 1),
          line_z = j, color = plot_theme.colors, linewidth = 2
          )                  
end

plot!(plt,
    xaxis = L"\ell", 
    yaxis = L"\ell(\ell+1)S_\ell",
    xscale = :log10
    )

plot!(twiny(),
      xaxis = L"k \; (h/\mathrm{Mpc})",
      xticks = (ℓ_ticks, k_ticklabels),
      xlims = (minimum(grid_data.ℓ), maximum(grid_data.ℓ))
     )

plot!(label = "Beyond BLAST", 
    legend = false, colorbar = true, colorbar_title = L"j", clims = (1, grid_data.Nkp),
    title = L"\ell(\ell+1)S_\ell^{gg}(k_1,k_2)",
    size=plot_theme.size_Cl, titlefontsize=20, 
    titleposition = :left;
    plot_theme.shared_style...
    )

plt

In [ ]:
println(size(S_lkk_gg))

In [ ]:
diagS = [diag(S_lkk_gg[:, :, i]) for i in 1:size(S_lkk_gg, 3)]

nshow = 100
idx = round.(Int, range(1, size(S_lkk_gg, 3), length=nshow))

p = plot(
    xlabel = L"k \; (h/\mathrm{Mpc})",
    ylabel = L"S_\ell(k,k)",
    title  = L"S_\ell^{gg}(k,k)\ \mathrm{for\ different}\ \ell\ \mathrm{values}",
    colorbar_title = L"j",
    clims = (1, grid_data.Nkp),                   
    legend = false,
    colorbar = true,
    lw = 2,
    size=plot_theme.size_Cl; plot_theme.shared_style...
)

for ii in idx
    plot!(p, grids.kp_grid, diagS[ii]/maximum(diagS[ii]), line_z = ii, color = plot_theme.colors;
        label = L"\ell = %$(grid_data.ℓ[ii])")
end

p

---